In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV,StratifiedKFold
from sklearn.metrics import roc_auc_score , log_loss

from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
import os
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore",category = ConvergenceWarning)

In [2]:
# !pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 
  

# print(cirrhosis_patient_survival_prediction.metadata) 
  

# print(cirrhosis_patient_survival_prediction.variables) 


In [4]:
y

,Status
0,D
1,C
2,D
3,D
4,CL
...,...
413,D
414,C
415,C
416,C


# We have to look at every column

In [5]:
X.columns

Index(['Drug', 'Age', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema',
       'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT',
       'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage'],
      dtype='str')

### Because Our data contains Too much NaNN values not nan values so we have replaved that NaNN to nan

In [6]:
X = X.replace(['NaNN','nan','nann','nann'], np.nan)

In [7]:
X.dtypes

Drug                 str
Age                int64
Sex                  str
Ascites              str
Hepatomegaly         str
Spiders              str
Edema                str
Bilirubin        float64
Cholesterol          str
Albumin          float64
Copper               str
Alk_Phos         float64
SGOT             float64
Tryglicerides        str
Platelets            str
Prothrombin      float64
Stage            float64
dtype: object

### Some columns odes contain numerical values but it is in string format in df so we convert it into float

In [8]:
X['Cholesterol'].astype(float)
X['Copper'].astype(float)
X['Tryglicerides'].astype(float)
X['Platelets'].astype(float)

0      190.0
1      221.0
2      151.0
3      183.0
4      136.0
       ...  
413    174.0
414    180.0
415    143.0
416    269.0
417    350.0
Name: Platelets, Length: 418, dtype: float64

In [9]:
X

,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261,2.60,156,1718.0,137.95,172,190,12.2,4.0
1,D-penicillamine,20617,F,N,Y,Y,N,1.1,302,4.14,54,7394.8,113.52,88,221,10.6,3.0
2,D-penicillamine,25594,M,N,N,N,S,1.4,176,3.48,210,516.0,96.10,55,151,12.0,4.0
3,D-penicillamine,19994,F,N,Y,Y,S,1.8,244,2.54,64,6121.8,60.63,92,183,10.3,4.0
4,Placebo,13918,F,N,Y,Y,N,3.4,279,3.53,143,671.0,113.15,72,136,10.9,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,NaN,24472,F,NaN,NaN,NaN,N,1.2,NaN,2.96,NaN,NaN,NaN,NaN,174,10.9,3.0
414,NaN,14245,F,NaN,NaN,NaN,N,0.9,NaN,3.83,NaN,NaN,NaN,NaN,180,11.2,4.0
415,NaN,20819,F,NaN,NaN,NaN,N,1.6,NaN,3.42,NaN,NaN,NaN,NaN,143,9.9,3.0
416,NaN,21185,F,NaN,NaN,NaN,N,0.8,NaN,3.75,NaN,NaN,NaN,NaN,269,10.4,3.0


In [10]:
num_imputer = SimpleImputer(strategy='mean').set_output(transform='pandas')
cat_imputer = SimpleImputer(strategy='constant',fill_value='Unknown').set_output(transform='pandas')

std_scaler = StandardScaler()

mm_scaler = MinMaxScaler()
knn = KNeighborsClassifier()
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
ct_impute = make_column_transformer((cat_imputer,make_column_selector(dtype_include=object)),
                                (num_imputer,make_column_selector(dtype_include=['int64','float64'])),
                                verbose_feature_names_out=False).set_output(transform='pandas')



In [11]:
ohe = OneHotEncoder(sparse_output=False,drop='first').set_output(transform='pandas')

ct_ohe = make_column_transformer((ohe,make_column_selector(dtype_include='object')),
                                ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                                verbose_feature_names_out=False).set_output(transform='pandas')

pipe = Pipeline([('IMP',ct_impute),('OHE',ct_ohe),('SCL',std_scaler),('KNN',knn)])
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('IMP', ...), ('OHE', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('simpleimputer-1', ...), ('simpleimputer-2', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g.

In [12]:
params = {'IMP__simpleimputer-1__strategy':['constant','most_frequent'],
          'IMP__simpleimputer-2__strategy':['mean','median'],
         'SCL':[std_scaler,mm_scaler,None],
         'KNN__n_neighbors':np.arange(1,11)
         }

In [13]:
gcv = GridSearchCV(pipe,param_grid=params,cv = kfold,n_jobs = -1,scoring='neg_log_loss')
gcv.fit(X,y['Status'])
print('Best Param : ',gcv.best_params_)
print('Best Score : ',gcv.best_score_)

Best Param :  {'IMP__simpleimputer-1__strategy': 'constant', 'IMP__simpleimputer-2__strategy': 'mean', 'KNN__n_neighbors': np.int64(1), 'SCL': StandardScaler()}
Best Score :  nan


C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


^C
  Using cached lxml-6.1.3-cp314-cp314-win_amd64.whl.metadata (3.4 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-win_amd64.whl.metadata (21 kB)
  Using cached propcache-0.5.4-cp314-cp314-win_amd64.whl.metadata (25 kB)
  Using cached yarl-1.25.1-cp314-cp314-win_amd64.whl.metadata (115 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
